# 06 - Microglia subset の subclustering ＋ subtype 自動アノテーション

05 で作成・保存した探索用 `adata_pca`
（`inner_logexpr_hvg_pca_umap_harmony_cluster_annotation_check.h5ad`）を読み込み、
**microglia だけを抜き出して再クラスタリング**し、microglia subtype マーカー遺伝子の
発現量を比較して **新しい列 `microglia_subtype_auto`** に自動アノテーションを書き込む。

最後に、(1) クラスタで色付けした UMAP、(2) アノテーション後の UMAP、(3) 各 subtype
ディレクトリにマーカー遺伝子を発現量で色付けした UMAP、を保存する。

## subtype マーカー（カテゴリ＝出力ディレクトリ）

| ディレクトリ | マーカー遺伝子 |
|---|---|
| `Homeostatic`   | P2ry12, Tmem119, Cx3cr1, Sall1 |
| `DAM_activated` | Apoe, Trem2, Tyrobp, Gpnmb, Lpl, Cst7, Cd68 |
| `Complement`    | C1qa, C1qb, C1qc |

## 重要な制約（必ず読むこと）

- 入力 `adata_pca` は **HVG 3000 遺伝子のみ**（`.raw` なし）。そのため上記 14 マーカーの
  うち **HVG に残っている 7 つ**しか発現で色付け・スコア化できない。
  - 利用可: Tmem119, Cx3cr1, Apoe, Trem2, Tyrobp, Lpl, Cst7
  - **欠落: P2ry12, Sall1, Gpnmb, Cd68, C1qa, C1qb, C1qc**
  - 特に **Complement (C1qa/b/c) は 3 つとも欠落**しており、Complement subtype は
    本ノートブックでは scoring・割り当てができない（割り当て候補から除外される）。
- 全マーカーで解析するには、full-gene の merged h5ad
  (`data/merged_h5ad/merged_qc_original_scale_inner.h5ad` / `outer.h5ad`) が必要。
  これは現在ディスク上に存在しないため、raw からパイプライン (01→04d) を再生成するか、
  merged h5ad を復元する必要がある。
- 自動アノテーションは **provisional / exploratory**。既知 annotation・marker 発現・
  cluster marker genes と必ず照合して解釈する。


## セットアップ（import・パス・adata_pca 読み込み）

In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("../../").resolve()

# 05 で保存した adata_pca を探す（repo results を優先、無ければ ~/Downloads を見る）
PCA_FILENAME = "inner_logexpr_hvg_pca_umap_harmony_cluster_annotation_check.h5ad"
PCA_CANDIDATES = [
    ROOT / "results" / "check_merged_h5ad" / PCA_FILENAME,
    Path.home() / "Downloads" / "results" / "check_merged_h5ad" / PCA_FILENAME,
]
PCA_PATH = next((p for p in PCA_CANDIDATES if p.exists()), None)
assert PCA_PATH is not None, (
    "adata_pca h5ad が見つからない。候補:\n  " + "\n  ".join(str(p) for p in PCA_CANDIDATES)
)
print("loading adata_pca:", PCA_PATH)

OUT_DIR = ROOT / "results" / "microglia_subclustering"
MARKER_DIR = OUT_DIR / "markers"
for d in (OUT_DIR, MARKER_DIR):
    d.mkdir(parents=True, exist_ok=True)

adata_pca = sc.read_h5ad(PCA_PATH)
print(adata_pca)
print("X is sparse:", sp.issparse(adata_pca.X), " X max:", float(adata_pca.X.max()))

loading adata_pca: /home/suzuki/Learn/SMA/v2/results/check_merged_h5ad/inner_logexpr_hvg_pca_umap_harmony_cluster_annotation_check.h5ad
AnnData object with n_obs × n_vars = 277279 × 3000
    obs: 'cell_uid', 'source_accession', 'dataset_id', 'Condition', 'qc_preprocessing_state', 'qc_pass_stage1', 'qc_reason_stage1', 'n_nonzero_genes', 'n_nonzero_mt_genes', 'pct_mt_for_qc', 'qc_pass_integrated', 'qc_reason_integrated', 'qc_pass_final', 'meta_cell.type.refined', 'meta_microglia.subclusters', 'meta_provisional.microglia.subclusters', 'meta_astrocyte.subclusters', 'meta_oligodendrocyte.subclusters', 'Age', 'sex_GSE206330', 'cell_type_original_GSE206330', 'meta_sex', 'sex', 'cell_type', 'leiden_before_harmony_r05', 'leiden_before_harmony_r10', 'leiden_harmony_r05', 'score_marker_Motor_Neuron', 'score_marker_Neuron', 'score_marker_Astrocyte', 'score_marker_Oligodendrocyte', 'score_marker_OPC', 'score_marker_Microglia', 'score_marker_Macrophage', 'score_marker_Endothelial', 'score_marker_Per

## 共通ヘルパー

In [2]:
def san(s):
    """ファイル名用に英数以外を _ に置換。"""
    return re.sub(r"[^0-9A-Za-z]+", "_", str(s)).strip("_")


def savefig(path):
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close("all")


def run_leiden(adata, resolution, key_added):
    """新しめの scanpy では flavor='igraph' 推奨。失敗時は default にフォールバック。"""
    try:
        sc.tl.leiden(adata, resolution=resolution, key_added=key_added,
                     flavor="igraph", n_iterations=2, directed=False)
    except Exception as e1:
        print(f"  [warn] leiden(flavor=igraph) failed ({e1}); trying default leiden")
        sc.tl.leiden(adata, resolution=resolution, key_added=key_added)

## 1. microglia subset を抜き出す

05 (cell 68) と同じ「microglia-like 判定」を使う。複数の annotation 列のいずれかが
microglia を示す細胞を `any_microglia_like` とし、その subset を作る。

In [3]:
UNKNOWN_STRINGS = {
    "", "unknown", "unk", "unassigned", "unannotated", "unclassified", "undefined",
    "undetermined", "unspecified", "na", "n/a", "n.a.", "none", "null", "nan",
    "missing", "no data", "no_data", "not available", "not_available",
    "not applicable", "not_applicable", "not detected", "not_detected",
    "not recorded", "not_recorded", "not provided", "not_provided",
}


def is_unknown_like(s):
    x = s.astype("object")
    x_str = x.astype(str).str.strip().str.lower()
    return x.isna() | x_str.isin(UNKNOWN_STRINGS)


def is_microglia_col(s):
    return (
        ~is_unknown_like(s)
        & s.astype(str).str.strip().str.lower().str.contains("microglia", na=False)
    )


def has_numeric_or_digit(s):
    x = s.astype("object")
    x_str = x.astype(str).str.strip()
    return (
        ~is_unknown_like(s)
        & (pd.to_numeric(x, errors="coerce").notna()
           | x_str.str.contains(r"\d", regex=True, na=False))
    )


obs = adata_pca.obs
flags = pd.DataFrame(index=adata_pca.obs_names)

for c in ["meta_cell.type.refined", "cell_type",
          "cell_type_original_GSE206330", "auto_cell_type_marker"]:
    flags[f"{c}_is_microglia"] = (
        is_microglia_col(obs[c]).to_numpy(dtype=bool) if c in obs.columns else False
    )

flags["meta_microglia.subclusters_has_number"] = (
    has_numeric_or_digit(obs["meta_microglia.subclusters"]).to_numpy(dtype=bool)
    if "meta_microglia.subclusters" in obs.columns else False
)

MICRO_FLAGS = [
    "meta_cell.type.refined_is_microglia",
    "cell_type_is_microglia",
    "cell_type_original_GSE206330_is_microglia",
    "auto_cell_type_marker_is_microglia",
    "meta_microglia.subclusters_has_number",
]
flags["any_microglia_like"] = flags[MICRO_FLAGS].any(axis=1)
for c in flags.columns:
    adata_pca.obs[c] = flags[c].to_numpy(dtype=bool)

print("total cells:", adata_pca.n_obs)
print("any_microglia_like:", int(flags["any_microglia_like"].sum()),
      f"({100*flags['any_microglia_like'].mean():.2f}%)")

mask_micro = flags["any_microglia_like"].to_numpy(dtype=bool)
adata_micro = adata_pca[mask_micro, :].copy()

# subset 後に壊れた *_colors で plot が落ちないよう削除
for k in list(adata_micro.uns.keys()):
    if str(k).endswith("_colors"):
        del adata_micro.uns[k]

# 元の log-expression を別 layer に退避（PCA/scale で .X が変わっても発現色付けに使える）
adata_micro.layers["logexpr"] = adata_micro.X.copy()

print("adata_micro:", adata_micro.shape)
display(adata_micro.obs["dataset_id"].astype(str).value_counts())

total cells: 277279
any_microglia_like: 66850 (24.11%)
adata_micro: (66850, 3000)


dataset_id
GSE295514_sc_brain_rnls8_tdp43_rds                           44413
GSE206330_sc_cortex_sod1_facs_glia_soupx                      8544
GSE287569_sc_spinalcord_sod1_ripk1i                           6269
GSE208629_sc_spinalcord_sma                                   3006
GSE219201_sn_lumbar_spinalcord_sod1_phd1                      2296
GSE242939_sc_spinalcord_sod1_pf04457845                        713
GSE167198_sc_spinalcord_sod1_dropseq                           636
GSE167327_sc_spinalcord_optn_cd45_enriched_indrop              540
GSE178693_sc_brainstem_trigeminal_sod1_dropseq                 269
GSE173524_sn_lumbar_spinalcord_sod1                            103
GSE167331_sc_spinalcord_optn_facs_microglia_smartseq2_tpm       61
Name: count, dtype: int64

## 2. microglia subset を再クラスタリング

microglia 内で HVG を選び直し、PCA → Harmony (batch=`dataset_id`) → neighbors → UMAP →
Leiden を microglia 専用に計算する（microglia 特化の埋め込み）。
Harmony が使えない場合は補正なし PCA にフォールバックする。

In [ ]:
# microglia 内で HVG を選び直す（入力 .X は log-normalized）
n_hvg = min(2000, adata_micro.n_vars - 1)
sc.pp.highly_variable_genes(adata_micro, n_top_genes=n_hvg, flavor="seurat")

n_comps = int(min(50, int(adata_micro.var["highly_variable"].sum()) - 1, adata_micro.n_obs - 1))
sc.pp.pca(adata_micro, n_comps=n_comps, mask_var="highly_variable")
print("PCA done; n_comps =", n_comps)

# Harmony で dataset 間の batch を補正（microglia 専用）
use_rep = "X_pca"
try:
    import harmonypy as hm

    meta = adata_micro.obs[["dataset_id"]].copy()
    meta["dataset_id"] = meta["dataset_id"].astype(str)
    ho = hm.run_harmony(
        data_mat=adata_micro.obsm["X_pca"],
        meta_data=meta,
        vars_use=["dataset_id"],
        max_iter_harmony=10,
        random_state=0,
    )
    Z = ho.Z_corr
    if Z.shape == adata_micro.obsm["X_pca"].shape:
        Xh = Z
    elif Z.T.shape == adata_micro.obsm["X_pca"].shape:
        Xh = Z.T
    else:
        raise ValueError(f"unexpected Harmony shape {Z.shape}")
    adata_micro.obsm["X_pca_harmony"] = np.ascontiguousarray(Xh)
    use_rep = "X_pca_harmony"
    print("Harmony OK; use_rep =", use_rep)
except Exception as e:
    import traceback
    print("[WARN] Harmony skipped; use_rep = X_pca")
    traceback.print_exc()

sc.pp.neighbors(adata_micro, use_rep=use_rep)
sc.tl.umap(adata_micro)
adata_micro.obsm["X_umap_microglia"] = adata_micro.obsm["X_umap"].copy()

CLUSTER_KEY = "microglia_leiden_r05"
run_leiden(adata_micro, resolution=0.5, key_added=CLUSTER_KEY)
print("clusters:", adata_micro.obs[CLUSTER_KEY].nunique())
display(adata_micro.obs[CLUSTER_KEY].value_counts().sort_index())

PCA done; n_comps = 50


2026-06-20 16:45:32,385 - harmonypy - INFO - Running Harmony
2026-06-20 16:45:32,386 - harmonypy - INFO -   Parameters:
2026-06-20 16:45:32,387 - harmonypy - INFO -     max_iter_harmony: 10
2026-06-20 16:45:32,389 - harmonypy - INFO -     max_iter_kmeans: 4
2026-06-20 16:45:32,390 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-06-20 16:45:32,391 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-06-20 16:45:32,392 - harmonypy - INFO -     nclust: 100
2026-06-20 16:45:32,394 - harmonypy - INFO -     block_size: 0.05
2026-06-20 16:45:32,395 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-06-20 16:45:32,396 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-06-20 16:45:32,397 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-06-20 16:45:32,398 - harmonypy - INFO -     verbose: True
2026-06-20 16:45:32,400 - harmonypy - INFO -     random_state: 0
2026-06-20 16:45:32,401 - harmonypy - INFO -   Data: 50 PCs × 66850 cells
2026-06-20 16:45:32,402

Harmony OK; use_rep = X_pca_harmony
clusters: 7


microglia_leiden_r05
0      956
1      471
2    14497
3    27851
4    21127
5     1851
6       97
Name: count, dtype: int64

## 3. subtype マーカーの定義と利用可否

`adata_pca` は HVG 3000 のみのため、HVG に残るマーカーだけがスコア化・色付けできる。
どのマーカーが利用可能/欠落かを `marker_availability.csv` に保存する。
var_names は uppercase なので case-insensitive に解決する。

In [5]:
MARKER_SETS = {
    "Homeostatic":   ["P2ry12", "Tmem119", "Cx3cr1", "Sall1"],
    "DAM_activated": ["Apoe", "Trem2", "Tyrobp", "Gpnmb", "Lpl", "Cst7", "Cd68"],
    "Complement":    ["C1qa", "C1qb", "C1qc"],
}

upper2var = {str(v).upper(): v for v in adata_micro.var_names}

avail_rows = []
resolved = {}  # category -> [(orig_symbol, var_name), ...]
for cat, genes in MARKER_SETS.items():
    res = []
    for g in genes:
        vn = upper2var.get(g.upper())
        avail_rows.append({"category": cat, "marker": g,
                           "present_in_hvg": vn is not None, "var_name": vn or ""})
        if vn is not None:
            res.append((g, vn))
    resolved[cat] = res
    present = [g for g, _ in res]
    missing = [g for g in genes if g.upper() not in upper2var]
    print(f"{cat}: present={present}  missing={missing}")

marker_availability = pd.DataFrame(avail_rows)
marker_availability.to_csv(OUT_DIR / "marker_availability.csv", index=False)
display(marker_availability)

Homeostatic: present=['Tmem119', 'Cx3cr1']  missing=['P2ry12', 'Sall1']
DAM_activated: present=['Apoe', 'Trem2', 'Tyrobp', 'Lpl', 'Cst7']  missing=['Gpnmb', 'Cd68']
Complement: present=[]  missing=['C1qa', 'C1qb', 'C1qc']


,category,marker,present_in_hvg,var_name
0,Homeostatic,P2ry12,False,
1,Homeostatic,Tmem119,True,TMEM119
2,Homeostatic,Cx3cr1,True,CX3CR1
3,Homeostatic,Sall1,False,
4,DAM_activated,Apoe,True,APOE
5,DAM_activated,Trem2,True,TREM2
6,DAM_activated,Tyrobp,True,TYROBP
7,DAM_activated,Gpnmb,False,
8,DAM_activated,Lpl,True,LPL
9,DAM_activated,Cst7,True,CST7


## 4. subtype スコア化 ＋ 自動アノテーション（新しい列）

各 subtype の利用可能マーカーで `sc.tl.score_genes` を計算し、クラスタごとの平均スコアが
最も高い subtype をそのクラスタに割り当てる。結果を新しい列 `microglia_subtype_auto` に書く。

**注意**: 利用可能マーカーが 0 個の subtype（= Complement）は割り当て候補から除外される。

In [6]:
score_cols = {}   # category -> score column name
for cat, res in resolved.items():
    genes = [vn for _, vn in res]
    sname = f"score_{cat}"
    if genes:
        sc.tl.score_genes(adata_micro, genes, score_name=sname)
        score_cols[cat] = sname
    else:
        adata_micro.obs[sname] = np.nan
        print(f"[WARN] {cat}: 利用可能マーカー 0 -> scoring 不可（割り当て候補から除外）")

usable = list(score_cols.keys())
sname_list = [score_cols[c] for c in usable]

# クラスタ x subtype の平均スコア行列
cluster_scores = adata_micro.obs.groupby(CLUSTER_KEY, observed=True)[sname_list].mean()
cluster_scores.columns = usable
cluster_scores.to_csv(OUT_DIR / "cluster_subtype_score_matrix.csv")

# クラスタごとに最大スコアの subtype を割り当て
cl2sub = {}
rows = []
for cl, row in cluster_scores.iterrows():
    top = row.sort_values(ascending=False)
    label = str(top.index[0])
    delta = float(top.iloc[0] - top.iloc[1]) if len(top) > 1 else np.nan
    cl2sub[cl] = label
    rows.append({
        "cluster": cl,
        "n_cells": int((adata_micro.obs[CLUSTER_KEY] == cl).sum()),
        "assigned_subtype": label,
        "top_score": float(top.iloc[0]),
        "score_delta_top1_top2": delta,
        **{f"score_{c}": float(row[c]) for c in usable},
    })
subtype_annotation = pd.DataFrame(rows)
subtype_annotation.to_csv(OUT_DIR / "microglia_subtype_annotation_by_cluster.csv", index=False)

# 新しい列にアノテーションを書き込む
adata_micro.obs["microglia_subtype_auto"] = (
    adata_micro.obs[CLUSTER_KEY].map(cl2sub).astype("category")
)

print("annotated subtypes assignable from available markers:", usable)
print("(Complement は全マーカー欠落のため割り当て不可)")
display(subtype_annotation)
display(adata_micro.obs["microglia_subtype_auto"].value_counts())

[WARN] Complement: 利用可能マーカー 0 -> scoring 不可（割り当て候補から除外）
annotated subtypes assignable from available markers: ['Homeostatic', 'DAM_activated']
(Complement は全マーカー欠落のため割り当て不可)


,cluster,n_cells,assigned_subtype,top_score,score_delta_top1_top2,score_Homeostatic,score_DAM_activated
0,0,956,DAM_activated,1.091403,0.334289,0.757114,1.091403
1,1,471,Homeostatic,0.903505,0.331265,0.903505,0.572240
2,2,14497,DAM_activated,1.594571,0.764700,0.829871,1.594571
3,3,27851,Homeostatic,1.781228,1.100858,1.781228,0.680370
4,4,21127,Homeostatic,1.495569,0.529725,1.495569,0.965844
5,5,1851,Homeostatic,1.035201,0.396677,1.035201,0.638524
6,6,97,DAM_activated,1.027156,1.707348,-0.680192,1.027156


microglia_subtype_auto
Homeostatic      51300
DAM_activated    15550
Name: count, dtype: int64

## 5. UMAP 保存

1. クラスタ (Leiden) で色付け
2. 自動アノテーション `microglia_subtype_auto` で色付け
3. 各 subtype の score も参考に色付け

In [7]:
basis = "X_umap_microglia"  # microglia 専用 UMAP

# 1. クラスタ
sc.pl.embedding(adata_micro, basis=basis, color=CLUSTER_KEY,
                legend_loc="on data", show=False, title="microglia Leiden clusters")
savefig(OUT_DIR / "umap_microglia_clusters.png")

# 2. アノテーション後
sc.pl.embedding(adata_micro, basis=basis, color="microglia_subtype_auto",
                show=False, title="microglia subtype (auto)")
savefig(OUT_DIR / "umap_microglia_subtype_auto.png")

# 参考: subtype score
for cat in usable:
    sc.pl.embedding(adata_micro, basis=basis, color=score_cols[cat],
                    color_map="viridis", show=False, title=f"score: {cat}")
    savefig(OUT_DIR / f"umap_score_{san(cat)}.png")

print("saved cluster / annotation / score UMAPs to:", OUT_DIR)

saved cluster / annotation / score UMAPs to: /home/suzuki/Learn/SMA/v2/results/microglia_subclustering


## 5b. dataset_id / Condition で色付け

batch（`dataset_id`）と実験条件（`Condition`）で microglia UMAP を色付けして保存する。
あわせて cluster / subtype ごとの構成比 CSV も出力する。

In [8]:
# dataset_id / Condition で色付け（microglia 専用 UMAP 上）
META_KEYS = ["dataset_id", "Condition"]

for k in META_KEYS:
    if k not in adata_micro.obs.columns:
        print(f"[skip] {k} not in obs")
        continue
    # legend が長くなりがちなので凡例は右側に出す
    sc.pl.embedding(adata_micro, basis=basis, color=k, show=False, title=f"microglia: {k}")
    savefig(OUT_DIR / f"umap_microglia_{san(k)}.png")
    print("saved", OUT_DIR / f"umap_microglia_{san(k)}.png")

    # 構成比も CSV に残す（cluster / subtype ごと）
    for grp in [CLUSTER_KEY, "microglia_subtype_auto"]:
        ct = (adata_micro.obs.groupby([grp, k], observed=True).size()
              .unstack(fill_value=0))
        ct.to_csv(OUT_DIR / f"composition_{san(grp)}_by_{san(k)}.csv")

print("done dataset_id / Condition UMAPs.")

saved /home/suzuki/Learn/SMA/v2/results/microglia_subclustering/umap_microglia_dataset_id.png
saved /home/suzuki/Learn/SMA/v2/results/microglia_subclustering/umap_microglia_Condition.png
done dataset_id / Condition UMAPs.


## 6. 各 subtype ディレクトリにマーカー遺伝子発現の UMAP を保存

`markers/<category>/umap_<gene>.png` として、各マーカー遺伝子の発現量で色付けした UMAP を
保存する。欠落マーカーは `MISSING_markers.txt` に記録してスキップする。

In [9]:
# 発現色付けは退避した log-expression を使う
adata_micro.X = adata_micro.layers["logexpr"].copy()

for cat, genes in MARKER_SETS.items():
    cdir = MARKER_DIR / san(cat)
    cdir.mkdir(parents=True, exist_ok=True)

    res = resolved[cat]
    missing = [g for g in genes if g.upper() not in upper2var]
    if missing:
        (cdir / "MISSING_markers.txt").write_text(
            "以下は HVG-3000 の adata_pca に存在しないためスキップ:\n"
            + "\n".join(missing) + "\n",
            encoding="utf-8",
        )
        print(f"[warn] {cat}: missing (skipped) -> {missing}")

    for orig, vn in res:
        sc.pl.embedding(adata_micro, basis=basis, color=vn,
                        color_map="viridis", show=False, title=f"{orig}  [{cat}]")
        savefig(cdir / f"umap_{san(orig)}.png")
        print(f"  saved {cat}/{orig}")

print("done marker UMAPs.")

[warn] Homeostatic: missing (skipped) -> ['P2ry12', 'Sall1']


  saved Homeostatic/Tmem119
  saved Homeostatic/Cx3cr1
[warn] DAM_activated: missing (skipped) -> ['Gpnmb', 'Cd68']
  saved DAM_activated/Apoe
  saved DAM_activated/Trem2
  saved DAM_activated/Tyrobp
  saved DAM_activated/Lpl
  saved DAM_activated/Cst7
[warn] Complement: missing (skipped) -> ['C1qa', 'C1qb', 'C1qc']
done marker UMAPs.


## 6b. クラスタ構成の可視化（dataset_id / Condition 由来）

各クラスタがどの `dataset_id`・`Condition` 由来かを stacked bar で可視化する。
- fraction 版: クラスタ内の構成比（どの dataset/condition が多いか）
- count 版: 実際の細胞数

In [10]:
def stacked_composition(adata, cluster_key, group_key, outpath, normalize=True):
    """cluster_key ごとに group_key の構成を stacked bar で描く。"""
    ct = (adata.obs.groupby([cluster_key, group_key], observed=True).size()
          .unstack(fill_value=0))
    ct = ct.sort_index()
    mat = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0) if normalize else ct
    ax = mat.plot(kind="bar", stacked=True,
                  figsize=(max(7, 0.6 * len(mat)), 5), colormap="tab20", width=0.85)
    ax.set_xlabel(cluster_key)
    ax.set_ylabel("fraction of cells" if normalize else "n cells")
    if normalize:
        ax.set_ylim(0, 1)
    ax.set_title(f"{cluster_key} composition by {group_key}"
                 + (" (fraction)" if normalize else " (counts)"))
    ax.legend(title=group_key, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
    plt.tight_layout()
    savefig(outpath)


for ckey in [CLUSTER_KEY, "microglia_subtype_auto"]:
    for gkey in ["dataset_id", "Condition"]:
        if gkey not in adata_micro.obs.columns:
            print(f"[skip] {gkey} not in obs")
            continue
        stacked_composition(adata_micro, ckey, gkey,
                            OUT_DIR / f"composition_{san(ckey)}_by_{san(gkey)}_fraction.png",
                            normalize=True)
        stacked_composition(adata_micro, ckey, gkey,
                            OUT_DIR / f"composition_{san(ckey)}_by_{san(gkey)}_counts.png",
                            normalize=False)
        print(f"saved composition plots: {ckey} x {gkey}")

print("done composition bar plots.")

saved composition plots: microglia_leiden_r05 x dataset_id
saved composition plots: microglia_leiden_r05 x Condition


saved composition plots: microglia_subtype_auto x dataset_id
saved composition plots: microglia_subtype_auto x Condition
done composition bar plots.


## 6c. マーカー発現の dotplot / matrixplot

利用可能な subtype マーカーが各クラスタ・subtype でどの程度発現しているかを
dotplot（発現量＋発現細胞割合）と matrixplot（平均発現）で見る。
`.X` は log-normalized（section 6 で logexpr に戻し済み）。

In [11]:
# dotplot 用のマーカー辞書（カテゴリ -> 利用可能な var_name）。空カテゴリは除外。
dot_markers = {cat: [vn for _, vn in resolved[cat]] for cat in MARKER_SETS if resolved[cat]}
print("dotplot markers:", dot_markers)

for grp in [CLUSTER_KEY, "microglia_subtype_auto"]:
    # dotplot（var ごとに 0-1 スケール）
    try:
        sc.pl.dotplot(adata_micro, dot_markers, groupby=grp,
                      standard_scale="var", show=False)
        savefig(OUT_DIR / f"marker_dotplot_by_{san(grp)}.png")
    except Exception as e:
        print(f"[warn] dotplot {grp}: {e}"); plt.close("all")
    # matrixplot（平均発現、var ごとに z-score）
    try:
        sc.pl.matrixplot(adata_micro, dot_markers, groupby=grp,
                         standard_scale="var", show=False)
        savefig(OUT_DIR / f"marker_matrixplot_by_{san(grp)}.png")
    except Exception as e:
        print(f"[warn] matrixplot {grp}: {e}"); plt.close("all")
    # stacked violin
    try:
        sc.pl.stacked_violin(adata_micro, dot_markers, groupby=grp, show=False)
        savefig(OUT_DIR / f"marker_stacked_violin_by_{san(grp)}.png")
    except Exception as e:
        print(f"[warn] stacked_violin {grp}: {e}"); plt.close("all")
    print("saved marker expression plots for", grp)

print("done dotplot / matrixplot / violin.")

dotplot markers: {'Homeostatic': ['TMEM119', 'CX3CR1'], 'DAM_activated': ['APOE', 'TREM2', 'TYROBP', 'LPL', 'CST7']}
saved marker expression plots for microglia_leiden_r05
saved marker expression plots for microglia_subtype_auto
done dotplot / matrixplot / violin.


## 6d. DEG（rank_genes_groups, Wilcoxon）

各クラスタ、および subtype（Homeostatic / DAM_activated）の marker 遺伝子を
Wilcoxon 検定で同定する。結果は CSV と図に保存する。
- HVG 3000 の log-normalized `.X` を使う（全遺伝子ではない点に注意）。
- subtype DEG は 2 群なので Homeostatic vs DAM_activated の比較になる。

In [12]:
def run_deg(adata, groupby, key_added, csv_path, plot_prefix):
    """rank_genes_groups (wilcoxon) を実行し、全 group の結果 CSV と図を保存。"""
    if adata.obs[groupby].nunique() < 2:
        print(f"[skip] {groupby}: <2 groups")
        return None
    sc.tl.rank_genes_groups(adata, groupby=groupby, method="wilcoxon",
                            key_added=key_added)
    df = sc.get.rank_genes_groups_df(adata, group=None, key=key_added)
    df.to_csv(csv_path, index=False)
    # top 遺伝子の図
    try:
        sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False, key=key_added, show=False)
        savefig(OUT_DIR / f"{plot_prefix}_top20.png")
    except Exception as e:
        print(f"[warn] rank_genes plot {groupby}: {e}"); plt.close("all")
    # top 遺伝子の dotplot
    try:
        sc.pl.rank_genes_groups_dotplot(adata, n_genes=5, key=key_added,
                                        standard_scale="var", show=False)
        savefig(OUT_DIR / f"{plot_prefix}_dotplot.png")
    except Exception as e:
        print(f"[warn] rank_genes dotplot {groupby}: {e}"); plt.close("all")
    print(f"saved DEG for {groupby}: {csv_path.name}  (rows={len(df)})")
    return df


deg_cluster = run_deg(adata_micro, CLUSTER_KEY, "rank_genes_cluster",
                      OUT_DIR / "deg_by_cluster_wilcoxon.csv", "deg_by_cluster")

deg_subtype = run_deg(adata_micro, "microglia_subtype_auto", "rank_genes_subtype",
                      OUT_DIR / "deg_by_subtype_wilcoxon.csv", "deg_by_subtype")

# 各クラスタの top10 を見やすく表示
if deg_cluster is not None:
    top10 = (deg_cluster.sort_values(["group", "scores"], ascending=[True, False])
             .groupby("group", observed=True).head(10))
    top10.to_csv(OUT_DIR / "deg_by_cluster_top10.csv", index=False)
    display(top10.pivot_table(index=top10.groupby("group").cumcount(),
                              columns="group", values="names", aggfunc="first"))
print("done DEG.")

saved DEG for microglia_leiden_r05: deg_by_cluster_wilcoxon.csv  (rows=21000)
saved DEG for microglia_subtype_auto: deg_by_subtype_wilcoxon.csv  (rows=6000)


group,0,1,2,3,4,5,6
0,HMGB2,STMN1,CCL3,JUN,NAV2,SPTBN1,CCL2
1,H2AFZ,TUBB5,CCL4,EGR1,ELMO1,TSC22D1,IER3
2,STMN1,CCND2,FTH1,IER5,CAMK1D,EPAS1,IGFBP4
3,BIRC5,TMSB10,RPL19,BTG2,SLC9A9,PTN,MARCKSL1
4,MKI67,HMGN2,H2-D1,ZFP36,LDLRAD4,ID3,LGALS1
5,TUBB5,FABP5,H2-K1,JUNB,TMCC3,UTRN,IFITM2
6,CDCA8,HMGB3,B2M,FOS,MAML3,ABLIM1,F13A1
7,TOP2A,ELAVL3,CD83,CST3,PLCL1,SPOCK2,LYVE1
8,CKS2,MEIS2,IFI27L2A,KCTD12,INPP5D,LY6C1,SDC4
9,SMC2,HNRNPA1,RPL15,IER2,GAB2,ID1,APOE


done DEG.


## 7. microglia subset の保存（再利用用）

In [13]:
# .X を logexpr に戻した状態で保存（layers にも logexpr あり）
out_h5ad = OUT_DIR / "adata_microglia_subclustered.h5ad"
adata_micro.write_h5ad(out_h5ad)
print("saved:", out_h5ad)
print("\nobs new columns:")
print(" - microglia_leiden_r05 (cluster)")
print(" - microglia_subtype_auto (annotation)")
print(" - score_Homeostatic / score_DAM_activated (+ score_Complement=NaN)")

saved: /home/suzuki/Learn/SMA/v2/results/microglia_subclustering/adata_microglia_subclustered.h5ad

obs new columns:
 - microglia_leiden_r05 (cluster)
 - microglia_subtype_auto (annotation)
 - score_Homeostatic / score_DAM_activated (+ score_Complement=NaN)


## まとめ

- 05 の `adata_pca`（HVG 3000）から **microglia 66k 細胞**を抜き出し、microglia 専用に
  PCA → Harmony(dataset_id) → UMAP → Leiden で再クラスタリングした。
- subtype マーカー発現を比較し、新しい列 **`microglia_subtype_auto`** に provisional
  アノテーションを書き込んだ。
- 出力: `results/microglia_subclustering/`
  - UMAP: `umap_microglia_clusters.png` / `umap_microglia_subtype_auto.png`
    / `umap_microglia_dataset_id.png` / `umap_microglia_Condition.png`
  - クラスタ構成: `composition_<cluster|subtype>_by_<dataset_id|Condition>_{fraction,counts}.png` ＋ CSV
  - マーカー発現: `marker_dotplot_by_*.png` / `marker_matrixplot_by_*.png` / `marker_stacked_violin_by_*.png`
  - DEG: `deg_by_cluster_wilcoxon.csv` / `deg_by_subtype_wilcoxon.csv` / `deg_by_cluster_top10.csv`
    ＋ `deg_by_*_top20.png` / `deg_by_*_dotplot.png`
  - subtype markers: `markers/{Homeostatic,DAM_activated,Complement}/umap_<gene>.png`
  - 参照: `marker_availability.csv` / `cluster_subtype_score_matrix.csv`
    / `microglia_subtype_annotation_by_cluster.csv`
  - `adata_microglia_subclustered.h5ad`
- **制約**: HVG 3000 のため Complement (C1qa/b/c) と P2ry12/Sall1/Gpnmb/Cd68 は欠落。
  DEG も HVG 3000 内での比較。完全な解析には full-gene merged h5ad が必要。


adata_micro

adata_micro.obs

In [16]:
adata_micro.obs["microglia_leiden_r05"].unique()

['0', '1', '2', '3', '5', '4', '6']
Categories (7, object): ['0', '1', '2', '3', '4', '5', '6']